In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, rand, expr, avg
from datetime import datetime, timedelta
import random

# Create a synthetic e-commerce transactions dataset with 10,000 rows
num_rows = 10000

# Product categories and codes
product_categories = ['Electronics', 'Clothing', 'Home & Garden', 'Books', 'Sports', 'Toys']
product_codes = ['PROD-{:04d}'.format(i) for i in range(1, 101)]

# Generate the dataset using Spark
df = spark.range(0, num_rows) \
    .withColumn("transaction_id", expr("concat('TXN-', lpad(id, 8, '0'))")) \
    .withColumn("user_id", (rand() * 5000).cast("int")) \
    .withColumn("product_code", expr("concat('PROD-', lpad(cast(rand() * 100 + 1 as int), 4, '0'))")) \
    .withColumn("product_category", expr("array('Electronics', 'Clothing', 'Home & Garden', 'Books', 'Sports', 'Toys')[cast(rand() * 6 as int)]")) \
    .withColumn("quantity", (rand() * 5 + 1).cast("int")) \
    .withColumn("unit_price", (rand() * 200 + 5).cast("double")) \
    .withColumn("discount_percent", 
                when(rand() < 0.3, (rand() * 0.25 + 0.05).cast("double"))
                .otherwise(0.0)) \
    .withColumn("amount", expr("round(quantity * unit_price * (1 - discount_percent), 2)")) \
    .withColumn("payment_method", expr("array('Credit Card', 'Debit Card', 'PayPal', 'Gift Card', 'Bank Transfer')[cast(rand() * 5 as int)]")) \
    .withColumn("shipping_country", expr("array('US', 'UK', 'Canada', 'Germany', 'France', 'Australia', 'Japan')[cast(rand() * 7 as int)]")) \
    .withColumn("is_mobile", (rand() < 0.6).cast("boolean")) \
    .withColumn("timestamp", 
                expr("timestamp('2024-01-01 00:00:00') + make_interval(0, 0, 0, 0, 0, 0, cast(rand() * 31536000 as double))")) \
    .drop("id")

# Introduce intentional null values in the amount column (~5% nulls)
df = df.withColumn("amount", 
                   when(rand() < 0.05, None).otherwise(col("amount")))

# Show sample data
display(df.limit(20))

# Print schema
print("\nDataset Schema:")
df.printSchema()

# Print summary statistics
print(f"\nTotal rows: {df.count()}")
print(f"Null values in amount column: {df.filter(col('amount').isNull()).count()}")
print(f"Non-null values in amount column: {df.filter(col('amount').isNotNull()).count()}")
print(f"Transactions with discount: {df.filter(col('discount_percent') > 0).count()}")
print(f"Average transaction amount: ${df.select(avg('amount')).first()[0]:.2f}" if df.filter(col('amount').isNotNull()).count() > 0 else "N/A")